# Rol 7. Responsable de operaciones y cumplimiento (Mejía)

**Task 1.7.** En este notebook se evalúa qué puede absorber la operación de cumplimiento con las alertas del piloto. Se calculan la capacidad, los costos y los casos detectados, y se redacta la sección 7 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

from src.operations import team_capacity, review_cost, detected_cases

## 1. Capacidad y costo

In [2]:
capacity = team_capacity(cd.ANALYSTS, cd.ALERTS_PER_ANALYST)
pilot_cost = review_cost(cd.PILOT_DAILY_ALERTS, cd.REVIEW_COST_USD)
capacity_cost = review_cost(capacity, cd.REVIEW_COST_USD)
print(f"Capacidad diaria: {capacity} alertas | costo: USD {capacity_cost:,.0f}")
print(f"Alertas del piloto: {cd.PILOT_DAILY_ALERTS} | costo de revisarlas todas: USD {pilot_cost:,.0f}")
print(f"Alertas por encima de la capacidad: {cd.PILOT_DAILY_ALERTS - capacity} ({cd.PILOT_DAILY_ALERTS / capacity:.2f} veces la capacidad)")

Capacidad diaria: 40 alertas | costo: USD 480
Alertas del piloto: 250 | costo de revisarlas todas: USD 3,000
Alertas por encima de la capacidad: 210 (6.25 veces la capacidad)


### a. Capacidad frente a alertas

El equipo puede revisar 40 alertas al día (8 analistas por 5 alertas), con un costo de USD 480. El piloto generó 250 alertas diarias, 6.25 veces la capacidad. Revisarlas todas costaría USD 3,000 diarios y dejaría 210 alertas diarias sin revisar con el equipo actual.

## 2. Casos detectados en el piloto

In [3]:
detected = detected_cases(cd.PILOT_DAILY_ALERTS, cd.PILOT_CONFIRMED_RATE)
print(f"Casos confirmados por día en el piloto: {detected:.0f}")
print(f"Fracción de los {cd.EXPECTED_DAILY_CASES} casos esperados: {detected / cd.EXPECTED_DAILY_CASES:.1%}")

Casos confirmados por día en el piloto: 10
Fracción de los 12 casos esperados: 83.3%


### b. Casos detectados

De las 250 alertas diarias, cerca del 4% resultó confirmado, es decir, 10 casos diarios. Esto representa el 83.3% de los 12 casos esperados por día. Esa cifra solo se alcanza si se revisan las 250 alertas.

## 3. Efecto de revisar solo lo que cabe en la capacidad

Se calcula cuántos casos se detectarían si el equipo revisara 40 alertas tomadas al azar entre las del piloto, y se compara el costo de ampliar la revisión con el costo de un caso no detectado.

In [4]:
reviewed_random = detected_cases(capacity, cd.PILOT_CONFIRMED_RATE)
extra_cost = review_cost(cd.PILOT_DAILY_ALERTS - capacity, cd.REVIEW_COST_USD)
print(f"Casos detectados si se revisan {capacity} alertas al azar del piloto: {reviewed_random:.1f}")
print(f"Costo diario de revisar las {cd.PILOT_DAILY_ALERTS - capacity} alertas restantes: USD {extra_cost:,.0f}")
print(f"Revisiones que equivalen al costo de un caso no detectado: {cd.MISSED_CASE_COST_USD / cd.REVIEW_COST_USD:,.0f}")

Casos detectados si se revisan 40 alertas al azar del piloto: 1.6
Costo diario de revisar las 210 alertas restantes: USD 2,520
Revisiones que equivalen al costo de un caso no detectado: 4,167


### c. Decisión operativa

Si el equipo revisa solo 40 alertas al azar, detecta 1.6 casos diarios en lugar de 10. Revisar las 210 alertas restantes cuesta USD 2,520 diarios, mientras que un solo caso no detectado cuesta lo mismo que 4,167 revisiones. Por ello se propone lo siguiente.

1. **No operar con U2 y el equipo actual**, porque la mayoría de alertas quedaría sin revisar.
2. **Ordenar la cola de alertas por puntaje de riesgo** para que las 40 revisiones diarias se dediquen a las alertas más probables, y no al azar.
3. **Ajustar el umbral y la capacidad en conjunto** para que las alertas diarias quepan en la capacidad. La elección final se toma con la tabla de umbrales a prevalencia real (Task 2.2).
4. **Ampliar el equipo de forma temporal** si el umbral que cabe en la capacidad deja demasiados casos sin detectar, porque el costo de revisar es mucho menor que el de no detectar.

### d. Sección 7 del memo

> **7. Operaciones.** Con el umbral U2, el piloto generó 250 alertas diarias frente a una capacidad de 40, es decir, 6.25 veces lo que el equipo puede revisar. Revisarlas todas cuesta USD 3,000 diarios y detecta 10 de los 12 casos esperados (83.3%), pero revisar 40 al azar detecta solo 1.6. Se recomienda no operar con U2 y el equipo actual, ordenar la cola por puntaje y ajustar umbral y capacidad para que las alertas quepan. Un caso no detectado (USD 50,000) cuesta lo mismo que 4,167 revisiones, lo que justifica ampliar la capacidad.